## Question B.2.
* Perform lemmatization using NLTK lemmatizer. Count the number of
unique words/tokens before and after lemmatization

In [1]:
# Import the necessary libraries
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import unicodedata
import string
import re

In [2]:
# Import the corpus
corpus = pd.read_csv('./Data.csv')

In [3]:
# Check the first 5 documents
corpus.head()

,Data
0,Watch or listen live weekdays at 8:30am MT at ...
1,Watch or listen live weekdays at 8:30am MT at ...
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
4,"Journalist, publisher of Rebel News — telling ..."


In [4]:
# Function to clean corpus
def preprocess(word):
    # Remove URLs
    word = re.sub(r'http\S+|www\S+|https\S+', '', word, flags=re.MULTILINE)
    # Remove '@' and hash '#' tags
    word = re.sub(r'@\w+', '', word)
    word = re.sub(r'#', '', word)
    word = re.sub(r'\brt\b', '', word, flags=re.IGNORECASE)
    # normalize unicode to ascii
    word = unicodedata.normalize('NFKD', word).encode('ascii', 'ignore').decode('utf-8')
    # Remove standalone digits
    word = re.sub(r'\b\d+\b', '', word)
    # Casefold
    word = word.casefold()
    # Normalize whitespace
    word = ' '.join(word.split())
    # Remove some special characters while maintaining necessary ones
    word = re.sub(r"[^\w\s'-]", '', word)

    # handle slashes between words
    if '/' in word:
        # first split to maintain known combinations
        if word.lower() in {'a/b', 'w/d', 'w/o', 'b/c'}:
            return word.lower()
        # second split
        parts = word.split('/')
        if len(parts) == 2 and len(parts[0]) < 8 and len(parts[1]) < 8:
            return '_'.join(parts)  # mountain_river
        # third split for other cases
        return ' '.join(parts)
    
    return word

In [5]:
# Apply cleaning to corpus
corpus['clean_data'] = corpus['Data'].apply(preprocess)

In [6]:
# Tokenize
corpus['tokens'] = corpus['clean_data'].apply(word_tokenize)

In [7]:
# Total count of words before lemmatization
all_tokens = [token for lst in corpus['tokens'] for token in lst]
num_unique_before = len(set(all_tokens))

print(f'Number of unique words before lemmatization: {num_unique_before}')

Number of unique words before lemmatization: 7644


In [8]:
# Lemmatizer initialization
lemmatizer = WordNetLemmatizer()

In [9]:
# Lemmatization function
def lemmatize_text(word):
    tokens = word_tokenize(word)
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmatized_tokens)

In [10]:
# Count unique words after lemmatization
corpus['lemmas'] = corpus['clean_data'].apply(lemmatize_text)
all_lemmas = ' '.join(corpus['lemmas']).split()
num_unique_after = len(set(all_lemmas))

print(f'Number of unique words before lemmatization: {num_unique_after}')

Number of unique words before lemmatization: 7097


In [11]:
print(f"Reduction: {num_unique_before - num_unique_after} words")

Reduction: 547 words


### Prompt
I need a function to properly handle slashes in my dataframe. I prefer a python solution to this problem